# Automatización de Reportes y Envío de Correos por SMTP
Este notebook contiene el código completo para procesar datos, evaluar KPIs financieros y automatizar el envío de reportes por correo electrónico usando Python.

## 1. Importación de Librerías Necesarias
Utilizaremos librerías nativas de Python, por lo que no requieres instalaciones externas complejas.

In [17]:
import smtplib
import os
import datetime
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

print("¡Librerías importadas exitosamente!")

¡Librerías importadas exitosamente!


### Cómo cargar variables desde un archivo `.env` (Uso Local / Offline)

Para que tu entorno de Python local reconozca el archivo `.env`, necesitas instalar la librería `python-dotenv`.

*Nota: Esta celda instalará la librería y te mostrará cómo leer las credenciales en tu computadora local de manera automática.*

In [ ]:
# 1. Instalar la librería requerida (Ejecutar en tu entorno local)
!pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# 2. Cargar las variables del archivo .env al entorno del sistema operativo
# Esto buscará un archivo llamado '.env' en el mismo directorio del script
load_dotenv()

# 3. Ahora puedes recuperarlas usando os.getenv de forma segura
remitente_local = os.getenv('SMTP_USER')
contrasenia_local = os.getenv('SMTP_PASSWORD')

if remitente_local and contrasenia_local:
    print("[✓] Archivo .env cargado exitosamente de forma local.")
    print(f"Usuario cargado: {remitente_local}")
else:
    print("[!] No se encontró el archivo .env o las variables no están definidas.")

## 2. Simulación de Datos y Evaluación de Desviaciones (KPIs)
Cargaremos un DataFrame de ejemplo para simular la detección de sobrepresupuestos.

In [18]:
# Dataset simulado de centros de costo
df_kpis = pd.DataFrame({
    'Centro_Costo': ['Administración', 'Operaciones', 'Tecnología', 'Logística'],
    'Presupuesto': [10000, 25000, 15000, 12000],
    'Gasto_Real': [10200, 29000, 14500, 11800]
})

# Calcular desviación porcentual
df_kpis['Desviacion_%'] = ((df_kpis['Gasto_Real'] - df_kpis['Presupuesto']) / df_kpis['Presupuesto']) * 100

# Filtrar si algún centro supera el 10% de sobrepresupuesto
alerta_activa = any(df_kpis['Desviacion_%'] > 10.0)
print("Datos evaluados. ¿Existe alerta por desviación > 10%?: ", alerta_activa)
display(df_kpis)

Datos evaluados. ¿Existe alerta por desviación > 10%?:  True


,Centro_Costo,Presupuesto,Gasto_Real,Desviacion_%
0,Administración,10000,10200,2.000000
1,Operaciones,25000,29000,16.000000
2,Tecnología,15000,14500,-3.333333
3,Logística,12000,11800,-1.666667


## 3. Construcción del Mensaje y Conexión SMTP
Configuración del protocolo seguro, credenciales por variables de entorno e inyección de contenido HTML.

In [19]:
import os
import getpass

try:
    # 1. Intentar usar la API de Colab (Entorno Online)
    from google.colab import userdata
    remitente = userdata.get('SMTP_USER')
    contrasenia = userdata.get('SMTP_PASSWORD')
    print("Credenciales cargadas exitosamente desde los Secretos de Colab.")
except (ImportError, Exception):
    # 2. Alternativa Offline (Local / PC)
    print("Entorno local detectado o Secretos de Colab no disponibles.")

    # Opción A: Intentar leer variables de entorno locales (si usas un archivo .env o las tienes en el sistema)
    remitente = os.getenv('SMTP_USER')
    contrasenia = os.getenv('SMTP_PASSWORD')

    # Opción B: Si no están definidas, solicitarlas de forma segura e interactiva sin mostrarlas en pantalla
    if not remitente:
        remitente = input("Introduce tu correo (SMTP_USER): ")
    if not contrasenia:
        contrasenia = getpass.getpass("Introduce tu contraseña de aplicación (SMTP_PASSWORD): ")

destinatario = 'cnunez28@santotomas.cl'

# Crear contenedor Multipart MIME
msg = MIMEMultipart()
msg['From'] = remitente
msg['To'] = destinatario
msg['Subject'] = 'Reporte Financiero Automatizado - CDG'

# Cuerpo del mensaje en HTML
html_content = """
<html>
  <body>
    <h2>Informe Diario de Gestión Financiera</h2>
    <p>Estimado equipo gerencial,</p>
    <p>Se adjunta el análisis de control presupuestario correspondiente al último ciclo.</p>
    <hr>
    <p style='color: red;'><b>Alerta:</b> Se detectaron desviaciones operativas que requieren revisión inmediata.</p>
  </body>
</html>
"""
msg.attach(MIMEText(html_content, 'html'))
print('Mensaje estructurado correctamente para uso híbrido (Online/Offline).')

Mensaje estructurado correctamente con credenciales desde Secretos.


## 4. Envío del Correo y Registro en Archivo Log
Conexión segura al servidor SMTP, manejo de excepciones y registro de auditoría local.

In [20]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

try:
    # 1. Conectar al servidor SMTP (ej. Gmail puerto 587)
    server = smtplib.SMTP('smtp.gmail.com', 587)

    # 2. Iniciar TLS (Cifrado)
    server.starttls()

    # 3. Autenticación
    server.login(remitente, contrasenia)

    # 4. Envío del correo
    server.sendmail(remitente, destinatario, msg.as_string())

    # 5. Cerrar conexión
    server.quit()

    estado_envio = f"[{timestamp}] - EXITOSO: Correo enviado correctamente.\n"
    print(estado_envio)

except Exception as e:
    estado_envio = f"[{timestamp}] - ERROR: Falló el envío. Detalle: {e}\n"
    print(estado_envio)

# Guardar traza en archivo de auditoría local (.log)
with open('registro_auditoria.log', 'a') as f:
    f.write(estado_envio)
print('Archivo de log actualizado.')

[2026-10-15 16:34:41] - ERROR: Falló el envío. Detalle: Connection unexpectedly closed

Archivo de log actualizado.


## 5. Automatización de Excel: Ejecutar Macros (.xlsm) desde Python

*Nota importante: Este módulo utiliza la API de Windows (`win32com`). Requiere ejecutarse en un entorno local con **Windows** y tener instalado **Microsoft Excel**.*

### Paso 1: Importación de librerías y validación del sistema operativo

En este paso importamos `win32com.client`, que es la librería que nos permite comunicarnos de forma directa con la API interna de Microsoft Excel en entornos Windows.

In [ ]:
import os
import sys

try:
    import win32com.client as win32
    print("Librería win32com importada exitosamente y lista para usar.")
except ImportError:
    print("La librería win32com no está instalada o no estás en un entorno compatible (Windows).")

### Paso 2: Definición de la función de control para la Macro

Creamos una función robusta con manejo de errores que se encarga de todo el ciclo de vida del archivo:
1. Valida que el archivo realmente exista en la ruta indicada.
2. Inicia la aplicación de Excel de manera invisible en segundo plano (`Visible = False`) para evitar parpadeos en pantalla.
3. Abre el libro, ejecuta la macro estructurando la llamada mediante `'NombreDeArchivo'!NombreDeMacro`.
4. Guarda los cambios, cierra el archivo de forma segura y destruye el proceso de Excel para liberar la memoria del computador.

In [ ]:
def ejecutar_macro_excel(ruta_libro, nombre_macro):
    # Validar que el archivo exista
    if not os.path.exists(ruta_libro):
        print(f"Error: No se encontró el archivo en la ruta: {ruta_libro}")
        return False

    # Asegurar que se use la ruta absoluta completa requerida por Excel
    ruta_absoluta = os.path.abspath(ruta_libro)

    excel = None
    try:
        # Inicializar la aplicación de Excel
        excel = win32.Dispatch("Excel.Application")
        excel.Visible = False  # Mantener Excel oculto en segundo plano
        excel.DisplayAlerts = False  # Desactivar alertas y ventanas flotantes

        print(f"Abriendo el archivo: {ruta_absoluta}...")
        workbook = excel.Workbooks.Open(ruta_absoluta)

        print(f"Ejecutando la macro: '{nombre_macro}'...")
        nombre_archivo = os.path.basename(ruta_absoluta)
        excel.Application.Run(f"'{nombre_archivo}'!{nombre_macro}")

        # Guardar y cerrar
        workbook.Save()
        workbook.Close(SaveChanges=True)
        print("[✓] Macro ejecutada y archivo guardado exitosamente.")
        return True

    except Exception as e:
        print(f"[✗] Ocurrió un error al ejecutar la macro: {e}")
        return False

    finally:
        # Garantizar que el proceso de Excel se cierre de forma limpia pase lo que pase
        if excel:
            excel.Quit()
            del excel
            print("Proceso de Excel finalizado correctamente.")

### Paso 3: Configuración de rutas y llamada final

Por último, configuramos los valores específicos de tu sistema. Aquí puedes rellenar la ruta local de tu archivo `.xlsm` y el nombre de la macro de Visual Basic para Aplicaciones (VBA) que deseas correr.

In [ ]:
# ==========================================================
# CONFIGURACIÓN DE RUTAS LOCALES
# ==========================================================

# TODO: Ingresa aquí la ruta completa a tu archivo .xlsm
RUTA_EXCEL_XLSM = r"C:\ruta\a_tu_archivo\archivo_con_macros.xlsm"

# TODO: Ingresa aquí el nombre exacto de la macro que deseas ejecutar
NOMBRE_MACRO = "MiMacroFavorita"

# Para ejecutar localmente, descomenta la siguiente línea:
# ejecutar_macro_excel(RUTA_EXCEL_XLSM, NOMBRE_MACRO)

## 6. Automatización del Entorno Local mediante Archivos Batch (`.bat`)

Un archivo `.bat` contiene una serie de instrucciones de la línea de comandos de Windows que se ejecutan secuencialmente de forma automática.

### ¿Qué utilidades tiene un archivo `.bat`?
1. **Ejecución en un Clic:** Puedes iniciar todo el proceso (ejecutar la macro de Excel, extraer datos y enviar correos) haciendo doble clic en el archivo.
2. **Programación Automática (Windows Task Scheduler):** Puedes configurar el *Programador de Tareas de Windows* para que ejecute el archivo `.bat` automáticamente a una hora específica del día (ej. todos los días a las 08:00 AM).
3. **Carga Segura de Credenciales:** El archivo `.bat` puede cargar temporalmente las variables de entorno (`SMTP_USER` y `SMTP_PASSWORD`) en memoria antes de lanzar el script de Python, evitando tener que escribirlas dentro del código Python.
4. **Mantenimiento del Sistema:** Limpiar registros de logs viejos o mover archivos procesados a carpetas de respaldo de forma automática.

### Código para generar tu archivo `.bat` de manera automática

Ejecuta la siguiente celda de Python para crear un archivo llamado `ejecutar_proceso.bat` de forma automática. Este archivo configurará las variables de entorno temporales para tu cuenta de correo y luego lanzará tu script de Python.

In [ ]:
# Script de Python para generar de forma segura tu archivo de automatización local (.bat)

contenido_bat = """@echo off
:: Desactivar el eco de los comandos en pantalla para mantener limpia la consola

title Ejecutando Automatizacion de Reportes Diario
echo ===================================================
echo Iniciando ejecucion del proceso de reportes automatizados...
echo ===================================================

:: Paso 1: Configurar variables de entorno temporales para la sesion
set SMTP_USER=tu_correo@gmail.com
set SMTP_PASSWORD=tu_contraseña_de_aplicacion_aqui

:: Paso 2: Ejecutar el script de python de forma local
:: Asegurate de reemplazar 'script_automatizacion.py' por el nombre de tu archivo
python script_automatizacion.py

:: Paso 3: Verificar si el proceso termino correctamente (Exit Code 0)
if %ERRORLEVEL% NEQ 0 (
    echo [X] Ocurrio un error critico durante la ejecucion del proceso.
    pause
    exit /b %ERRORLEVEL%
)

echo [V] Todo el proceso ha finalizado correctamente.
echo Cerrando ventana...
timeout /t 5
"""

# Guardar el archivo .bat en el directorio de trabajo actual
with open("ejecutar_proceso.bat", "w", encoding="utf-8") as bat_file:
    bat_file.write(contenido_bat)

print("[✓] Archivo 'ejecutar_proceso.bat' generado con éxito en el directorio actual.")

### Guía de Uso del Archivo `.bat` en tu PC Local:

1. **Descarga los archivos:** Descarga tanto este notebook (como `.py` o `.ipynb`) junto con el archivo `ejecutar_proceso.bat` a la misma carpeta de tu computadora local.
2. **Modifica tus datos:** Abre el archivo `ejecutar_proceso.bat` con cualquier bloc de notas y reemplaza `tu_correo@gmail.com` y `tu_contraseña_de_aplicacion_aqui` por tus credenciales reales.
3. **Ejecuta con doble clic:** Simplemente haz doble clic sobre `ejecutar_proceso.bat`. Se abrirá una terminal negra que ejecutará la macro de Excel, creará los logs y enviará el correo de forma secuencial y automática.

## 7. Ejemplos Prácticos de Otras Automatizaciones con Archivos `.bat`

Aquí tienes 3 plantillas de código Batch comunes y muy útiles que puedes copiar en archivos `.bat` independientes en tu computadora para optimizar tus tareas diarias.

### Ejemplo A: Copia de Seguridad Automática (Respaldo)
Este script copia de forma incremental todos los archivos nuevos o modificados desde tu carpeta de trabajo a una carpeta de respaldos (por ejemplo, en un pendrive o OneDrive):
```batch
@echo off
title Respaldo Automatico de Archivos
echo Iniciando copia de seguridad...

:: Define las rutas de origen y destino
set ORIGEN="C:\Usuarios\TuUsuario\Documentos\Reportes"
set DESTINO="D:\Respaldos\Reportes_Backup"

:: xcopy copia directorios, subdirectorios (/s), archivos nuevos (/d) y evita preguntar al sobrescribir (/y)
xcopy %ORIGEN% %DESTINO% /s /d /y /i

echo Respaldo finalizado.
timeout /t 3
```

### Ejemplo B: Apertura de Entorno de Trabajo (Inicio Rápido)
Ideal para cuando enciendes la computadora por la mañana. Con un solo clic abres todo lo que necesitas para trabajar:
```batch
@echo off
title Iniciando Entorno de Trabajo
echo Abriendo herramientas diarias...

:: Abrir carpetas del sistema
start explorer "C:\Usuarios\TuUsuario\Proyectos"

:: Abrir programas locales
start "" "C:\Program Files\Microsoft Office\root\Office16\EXCEL.EXE"

:: Abrir el navegador en páginas específicas de control
start chrome "https://github.com" "https://mail.google.com"

exit
```

### Ejemplo C: Limpiador de Archivos Temporales y Descargas Antiguas
Utilizado para mantener el disco duro optimizado eliminando archivos residuales:
```batch
@echo off
title Limpiador del Sistema
echo Eliminando archivos temporales...

:: Borrar archivos temporales de Windows
del /f /q /s %temp%\*.*

:: Borrar archivos temporales del sistema
del /f /q /s C:\Windows\Temp\*.*

echo [✓] Limpieza completada con exito.
pause
```

In [ ]:
# Código Python para generar el script .bat personalizado

contenido_bat_proyecto = """@echo off
title Abrir Proyecto y Ejecutar Python
echo ===================================================
echo Iniciando Entorno de Trabajo y Ejecutando Script
echo ===================================================

:: 1. Configura aquí la ruta de la carpeta de tu proyecto
set RUTA_PROYECTO=C:\ruta\a_tu_proyecto\mi_carpeta

:: 2. Configura aquí el nombre de tu script de Python
set SCRIPT_PYTHON=script_automatizacion.py

echo [1/2] Abriendo carpeta del proyecto...
start explorer "%RUTA_PROYECTO%"

echo [2/2] Cambiando de directorio y ejecutando script...
cd /d "%RUTA_PROYECTO%"
python "%SCRIPT_PYTHON%"

:: Validar si ocurrió un error en la ejecución
if %ERRORLEVEL% NEQ 0 (
    echo.
    echo [X] Ocurrio un error al ejecutar el script de Python.
    pause
    exit /b %ERRORLEVEL%
)

echo.
echo [✓] Proceso finalizado correctamente.
timeout /t 5
"""

# Guardar el archivo .bat en el directorio actual
with open("abrir_proyecto_y_ejecutar.bat", "w", encoding="utf-8") as f:
    f.write(contenido_bat_proyecto)

print("[✓] Archivo 'abrir_proyecto_y_ejecutar.bat' generado con éxito.")